# Notebook 02 - The Maths for ML (Only the Idea)

Goes with **Lesson 04**. Python does all the calculating. You just read and run.

## 1. Variables and functions

A machine learning model is just a function: input goes in, output comes out.

In [ ]:
def f(x):
    return 2 * x + 1

for x in [0, 1, 2, 3]:
    print(f"f({x}) = {f(x)}")

## 2. Vectors and matrices with numpy

A **vector** is a list of numbers. A **matrix** is a table of numbers.

In [ ]:
import numpy as np

# One student as a vector: [hours, attendance, sleep]
student = np.array([6.3, 46, 7.7])
print("Vector:", student)
print("Vector shape:", student.shape)

# Three students as a matrix (rows = students, columns = features)
students = np.array([
    [6.3, 46, 7.7],
    [1.3, 72, 4.2],
    [5.6, 90, 7.2],
])
print()
print("Matrix:")
print(students)
print("Matrix shape:", students.shape)

numpy can do maths on a whole vector at once. A very important operation is the **weighted sum** (also called the dot product). Models use it all the time: multiply each feature by a weight and add up.

In [ ]:
weights = np.array([4.0, 0.2, 1.5])   # how important each feature is

score = np.dot(student, weights)       # 6.3*4.0 + 46*0.2 + 7.7*1.5
print("Weighted sum for one student:", round(score, 2))

# The same for all three students at once (matrix times vector)
print("Weighted sums for all students:", students @ weights)

## 3. Describing data: mean, median, mode, variance, standard deviation

First the small example from the lesson.

In [ ]:
import statistics

marks = [40, 55, 55, 60, 70, 80, 100]

print("Mean:              ", round(statistics.mean(marks), 1))
print("Median:            ", statistics.median(marks))
print("Mode:              ", statistics.mode(marks))
print("Range:             ", max(marks) - min(marks))
print("Variance:          ", round(np.var(marks), 1))
print("Standard deviation:", round(np.std(marks), 1))

Now on the real student data.

In [ ]:
from pathlib import Path

# The data files are in the "data" folder next to "notebooks".
# In Google Colab, upload the CSV file and it will be found in the current folder.
DATA_DIR = Path("../data") if Path("../data").exists() else Path(".")
print("Reading data from:", DATA_DIR.resolve())

In [ ]:
import pandas as pd

df = pd.read_csv(DATA_DIR / "student_performance.csv")
scores = df["final_score"]

print("Mean final score:  ", round(scores.mean(), 1))
print("Median final score:", scores.median())
print("Mode final score:  ", scores.mode()[0])
print("Standard deviation:", round(scores.std(), 1))
print("Lowest and highest:", scores.min(), "and", scores.max())

## 4. Outliers change the mean, not the median

Watch what one extreme value does.

In [ ]:
salaries = [20, 22, 25, 24, 23]          # in thousands
print("Mean:", np.mean(salaries), " Median:", np.median(salaries))

salaries_with_ceo = salaries + [500]     # one very rich person joins
print("Mean:", round(np.mean(salaries_with_ceo), 1), " Median:", np.median(salaries_with_ceo))

The mean jumped from 22.8 to about 102, but the median barely moved. That is why the median is safer when there are outliers.

## 5. Probability

In [ ]:
p_pass = (df["result"] == "pass").mean()
print(f"Probability that a random student passed: {p_pass:.2f}")

with_extra = df[df["extra_classes"] == "yes"]
without_extra = df[df["extra_classes"] == "no"]
print(f"Probability of passing WITH extra classes:    {(with_extra['result'] == 'pass').mean():.2f}")
print(f"Probability of passing WITHOUT extra classes: {(without_extra['result'] == 'pass').mean():.2f}")

## 6. Correlation

Numbers from -1 to +1. Closer to +1 means "go up together".

In [ ]:
correlation = df.corr(numeric_only=True)["final_score"].sort_values(ascending=False)
print(correlation.round(2))

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(df["hours_studied"], df["final_score"], alpha=0.6)
plt.title("Hours studied vs final score (strong positive correlation)")
plt.xlabel("Hours studied per day")
plt.ylabel("Final score")
plt.show()

## 7. Optimisation: gradient descent, step by step

We want the computer to find the best line `score = a * hours + b`.

1. Start with a bad guess: `a = 0`, `b = 0`.
2. Measure the **loss** (how wrong we are). We use the average of the squared errors.
3. Find the slope (gradient) of the loss and take a small step **downhill**.
4. Repeat for many **epochs**.

In [ ]:
x = df["hours_studied"].to_numpy()
y = df["final_score"].to_numpy()

a, b = 0.0, 0.0          # starting guess
learning_rate = 0.01     # size of each step
epochs = 2000
loss_history = []

for epoch in range(epochs):
    prediction = a * x + b
    error = prediction - y
    loss = (error ** 2).mean()            # mean squared error
    loss_history.append(loss)

    # gradients: which way is "downhill" for a and for b
    grad_a = 2 * (error * x).mean()
    grad_b = 2 * error.mean()

    # take a small step downhill
    a = a - learning_rate * grad_a
    b = b - learning_rate * grad_b

    if epoch in [0, 1, 2, 10, 100, 500, epochs - 1]:
        print(f"epoch {epoch:4}:  loss = {loss:9.1f}   a = {a:5.2f}   b = {b:5.2f}")

In [ ]:
plt.plot(loss_history)
plt.title("The loss goes DOWN as the model learns")
plt.xlabel("Epoch")
plt.ylabel("Loss (mean squared error)")
plt.yscale("log")
plt.show()

plt.scatter(x, y, alpha=0.4, label="students")
xs = np.linspace(0, 10, 50)
plt.plot(xs, a * xs + b, color="red", label=f"learned line: score = {a:.1f} x hours + {b:.1f}")
plt.xlabel("Hours studied")
plt.ylabel("Final score")
plt.legend()
plt.show()

The computer **learned** that every extra hour of study adds about `a` marks. Nobody told it this number.

**Try it:** change `learning_rate` to `0.1` and run the two cells again. The loss explodes (becomes huge or `nan`) because the steps are too big. Then try `0.001` - it learns, but more slowly.

Go back to **Lesson 04** and do the **Check** section.